## Imports

In [ ]:
import xarray as xr

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from xhistogram.xarray import histogram
import matplotlib.colors as colors
import cartopy.crs as ccrs
import scipy
import matplotlib.patches as patches

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 16})

from matplotlib import cm, ticker
from matplotlib import colors as cols

## Open the data

In [ ]:
# data in persistent bucket
#target_url ='gs://leap-scratch/cspencerjones/hero-calc/masked_hist.zarr'



###target_url ='gs://leap-persistent/cspencerjones/hero-calc/compute/llc4320/masked_hist_48hr.zarr'
#target_url ='gs://leap-persistent/cspencerjones/hero-calc/compute/llc4320/masked_hist_renorm_48hr.zarr'
target_url ='gs://leap-persistent/cspencerjones/hero-calc/masked_hist_renorm_3hr.zarr'
#target_url2 = 'gs://leap-persistent/tomnicholas/hero-calc/compute/llc4320/vort_strain_div_histogram_coarsen_nan_padding_3_hourly.zarr'

In [ ]:
hist_ds = xr.open_dataset(target_url, engine="zarr", chunks={})
#hist_ds2 = xr.open_dataset(target_url2, engine="zarr", chunks={})

In [ ]:
hist_ds['histogram_vort_strain_div'].isel(time=0).nbytes / 1e9

In [ ]:
hist_ds

In [ ]:
from xarray.indexes import PandasIndex

# have to add the index ourselves manually for some reason
hist_ds = hist_ds.set_xindex('region_num', PandasIndex)
#hist_ds2 = hist_ds2.set_xindex('region_num', PandasIndex)

In [ ]:
hist_ds

## Correct PDF normalization

Division by 80 comes from 8 samples per day (3-hourly sampling of hourly data) over a 10-day period. 
This correction is currently required because I accidentally did `coarsen(time=8*10).sum(dim='time')` instead of `coarsen(time=8*10).mean(dim='time')` before saving to zarr.

In [ ]:
#hist_ds2['histogram_vort_strain_div'] = hist_ds2['histogram_vort_strain_div'] / 80

In [ ]:
hist_ds['histogram_vort_strain_div'].isel(time=slice(11,20)).time

In [ ]:
h = hist_ds['histogram_vort_strain_div']
h['center_lat'] = hist_ds.vertices_latitude.mean('vertices')
h['center_lon'] = hist_ds.vertices_longitude.mean('vertices')
max_vert = abs(hist_ds.vertices_longitude.diff('vertices')).max('vertices')
h['center_lon'] = h['center_lon'].where(max_vert<100,((hist_ds.vertices_longitude.where(max_vert>100).min('vertices') +hist_ds.vertices_longitude.where(max_vert>100).max('vertices')+360)/2))
h1 = h.mean('time')

In [ ]:

h1_2d = (h1.sum('div_bin').mean('region')/970).load()

div_2d = ((h1.div_bin*h1).mean('div_bin').mean('region')/970).load()
div_2d_masked = div_2d.where(h1_2d>10**-5)

In [ ]:


plt.figure(figsize=(7.5,7))
levels = 10**(np.arange(-5.5, -1.5, 0.01))

plt.subplot(211)
h1_2d.plot.contourf(x='vort_bin', vmax=0.02,norm=cols.SymLogNorm(1e-7), cmap='Reds',
                    vmin=10**-5.5,levels=levels,extend='both',cbar_kwargs={'ticks': [10**-5, 10**-4, 10**-3, 10**-2],'label':''})

h1_2d.plot.contour(x='vort_bin',levels=[1e-3, 1e-4, 1e-5], colors='k', alpha=.3)

#plt.contour(h1_2d.vort_bin, h1_2d.strain_bin, h1_2d, levels=[1e-5], colors='k', alpha=.3)



ax=plt.gca()

plt.plot(np.linspace(0,5), np.linspace(0,5),color='k',linewidth=1,linestyle='--')
plt.plot(np.linspace(-5,0), np.linspace(5,0),color='k',linewidth=1,linestyle='--')

#plt.plot(np.linspace(0,3.5), np.linspace(1.5,5),color='k',linewidth=0.5,linestyle=':')
#plt.plot(np.linspace(-3.5,0), np.linspace(5,1.5),color='k',linewidth=0.5,linestyle=':')
# Create a Rectangle patch
#rect = patches.Rectangle((0.8, 0.8), 5, 5, linewidth=1, edgecolor='k', facecolor='none',linestyle='--')

# Add the patch to the Axes
#ax.add_patch(rect)

## Create a Rectangle patch
#rect = patches.Rectangle((-0.5, 1), 1, 4, linewidth=1, edgecolor='k', facecolor='none',linestyle='--')

# Add the patch to the Axes
#ax.add_patch(rect)

x_ticks = np.arange(-5, 5)
y_ticks = np.arange(0, 5)

ax.set_xticks(x_ticks)

ax.set_yticks(y_ticks)


plt.xlim(-5,5)
plt.ylim(0,5)
plt.xlabel('')
plt.ylabel('$\sigma/f_{floor}$')
plt.rc('grid', color='black', alpha=.3)
plt.grid()


plt.title('(a) Vorticity-strain JPDF for whole dataset',fontsize=16)


plt.subplot(212)
levels = 10**(np.arange(-5.5, -1.5, 0.1))

#h1_2d.plot.contour(x='vort_bin',
#                    vmin=10**-5.5,levels=levels,colors='k',linewidths=0.1)

h1_2d.plot.contour(x='vort_bin',levels=[1e-3, 1e-4, 1e-5], colors='k', alpha=.3)

(div_2d*10**5).plot(x='vort_bin',vmax=1, norm=cols.SymLogNorm(1e-2), extend='both', cmap='RdBu_r')


plt.plot(np.linspace(0,5), np.linspace(0,5),color='k',linewidth=1,linestyle='--')
plt.plot(np.linspace(-5,0), np.linspace(5,0),color='k',linewidth=1,linestyle='--')

#plt.plot(np.linspace(0,3.5), np.linspace(1.5,5),color='k',linewidth=0.5,linestyle=':')
#plt.plot(np.linspace(-3.5,0), np.linspace(5,1.5),color='k',linewidth=0.5,linestyle=':')
ax=plt.gca()

# Create a Rectangle patch
#rect = patches.Rectangle((1, 1), 5, 5, linewidth=1, edgecolor='k', facecolor='none',linestyle='--')

# Add the patch to the Axes
#ax.add_patch(rect)

## Create a Rectangle patch
#rect = patches.Rectangle((-0.5, 1), 1, 4, linewidth=1, edgecolor='k', facecolor='none',linestyle='--')

# Add the patch to the Axes
#ax.add_patch(rect)

plt.xlim(-5,5)
plt.ylim(0,5)
plt.xlabel('$\zeta/f_{floor}$')
plt.ylabel('$\sigma/f_{floor}$')
plt.grid()

x_ticks = np.arange(-5, 5)
y_ticks = np.arange(0, 5)

ax.set_xticks(x_ticks)

ax.set_yticks(y_ticks)



plt.title('(b) Mean divergence in each bin (x$ 10^{-4}$/s)',fontsize=16)

plt.subplots_adjust(bottom=0.2,hspace=0.3)
plt.savefig('vort-strain-complete.png')